Miscellaneous stuff

If the same pattern is used in many function calls, it may be wise to precompile the pattern, mainly for efficiency reasons.

 This can be done using the compile(pattern, flags=0) function in the re module.

The function returns a so-called RE object.

The RE object has method versions of the functions found in module re.

The only difference is that the first parameter is not the pattern since the precompiled pattern is stored in the RE object.

In [1]:
import re

date_pattern=re.compile(r'\d{4}-\d{2}-\d{2}')
print(date_pattern)

re.compile('\\d{4}-\\d{2}-\\d{2}')


## Precompiling Regex Patterns — Simply

This is the same idea you've seen elsewhere: **do expensive setup once, reuse it many times** — instead of redoing the setup work on every single call.

---

### The Problem — Regex Patterns Need Interpretation Every Time

Under the hood, every time you call `re.search(pattern, s)`, Python has to first **parse the pattern string** into an internal instruction set, *then* run the search. If you call it 10,000 times with the same pattern, it re-parses the exact same pattern **10,000 times** — wasted, repeated work.

```python
import re

for line in huge_list_of_lines:
    re.search(r'\d{4}-\d{2}-\d{2}', line)   # pattern re-parsed EVERY iteration
```

---

### The Fix — `re.compile()` Parses Once

```python
date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

for line in huge_list_of_lines:
    date_pattern.search(line)                # already-parsed — just runs
```

Now the pattern is parsed **once**, stored in `date_pattern`, and every loop iteration reuses the ready-made version.

---

### What `re.compile()` Returns — an "RE Object"

```python
date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')
print(date_pattern)
# → re.compile('\\d{4}-\\d{2}-\\d{2}')
```

This is an **object** — same concept as everything else you've built objects with (match objects, function objects). It has the pattern **baked in**.

---

### The Key Rule — Same Functions, One Less Argument

The RE object has **method versions** of `search`, `match`, `findall`, `finditer`, `sub` — everything you already know! The only difference: **no pattern argument**, because the pattern already lives inside the object.

```python
# Module-level function — pattern passed every time:
re.search(r'\d+', s)
re.findall(r'\d+', s)
re.sub(r'\d+', '#', s)

# Compiled version — pattern already baked in, just call the method:
p = re.compile(r'\d+')
p.search(s)
p.findall(s)
p.sub('#', s)
```

**Side by side:**

| Module function | Compiled equivalent |
|---|---|
| `re.match(pattern, s)` | `p.match(s)` |
| `re.search(pattern, s)` | `p.search(s)` |
| `re.findall(pattern, s)` | `p.findall(s)` |
| `re.finditer(pattern, s)` | `p.finditer(s)` |
| `re.sub(pattern, repl, s)` | `p.sub(repl, s)` |

Same call, minus the pattern — the object *is* the pattern, plus behavior.

---

### Full Example

```python
import re

email_pattern = re.compile(r'\b[\w.]+@[\w.]+\.\w+\b')

texts = [
    "Contact: alice@example.com for details",
    "No email here",
    "Reach bob.smith@company.org anytime"
]

for t in texts:
    m = email_pattern.search(t)
    if m:
        print(m.group())
```

```
alice@example.com
bob.smith@company.org
```

Same `.search()`, `.group()` you already know — just called **on the compiled object** instead of the `re` module.

---

### A Familiar Analogy from Your Own Course Material

> Recall `defaultdict(list)` — you built the **empty structure once**, then reused it across many `.append()` calls without re-specifying "use a list" each time.
>
> `re.compile()` is the same shape: **configure once, use repeatedly.** The module function `re.search(pattern, s)` is like calling `list()` fresh every time you need an empty list — works, but wasteful if you do it constantly.

---

### When Does This Actually Matter?

For a handful of calls, the difference is invisible — Python even **caches** recently-used patterns internally, so casual one-off `re.search(...)` calls aren't as wasteful as it might sound. Precompiling earns its keep when:

- The **same pattern** is used **many times** (loops over large files, repeated validation)
- You want a **named, reusable** pattern object across your codebase — self-documenting too:

```python
YEAR = re.compile(r'^\d{4}$')

def is_valid_year(s):
    return bool(YEAR.match(s))
```

---

### The One-Sentence Summary

> `re.compile(pattern)` parses the pattern **once** into a reusable RE object; that object offers the same methods (`.search`, `.match`, `.findall`, `.finditer`, `.sub`) as the `re` module functions, just **without needing the pattern argument again** — because it's already stored inside the object. Precompile when the same pattern gets used repeatedly. 🎯

The details of matching operation can be specified using optional flags. These flags can be given either inside the pattern or as a parameter to the compile function. Some of the more common flags are given in the following table


| Flag | Description |
| :--- | :--- |
| `x` | Flag |
| `(?i)` | `re.IGNORECASE` |
| `(?m)` | `re.MULTILINE` |
| `(?s)` | `re.DOTALL` |



The elements on the left can appear anywhere in the pattern but preferably in the beginning. On the right there are attributes of the re module that can be given to the compile function as the second parameter

The IGNORECASE flag makes lower- and uppercase characters appear as equal. The ```MULTILINE``` flag makes the special characters ^ and $ match the beginning and end of each line in addition to the beginning and end of the whole string. These flags make ```\A``` differ from ```^```, and ```\Z``` differ from ```$```. The ```DOTALL``` flag makes the character class ```.``` (dot) also accept the newline character, in addition to all the other letters.

In [15]:
str='HELLO world'
print(re.search(r'(?i)hello',str))
print(re.search(r'hello',str,re.IGNORECASE))
print(re.search(r'hello|HELLO',str))

<re.Match object; span=(0, 5), match='HELLO'>
<re.Match object; span=(0, 5), match='HELLO'>
<re.Match object; span=(0, 5), match='HELLO'>


## Regex Flags — Simply

Flags let you **change the rules of matching** for an entire pattern — without rewriting the pattern itself. Same regex, different behavior.

---

### Two Ways to Set a Flag

**Way 1 — Inside the pattern**, as `(?x)` at the start:

```python
re.search(r'(?i)hello', 'HELLO world')
```

**Way 2 — As a parameter** to `compile` (or directly to `search`/`match`/etc.):

```python
p = re.compile(r'hello', re.IGNORECASE)
p.search('HELLO world')

# or directly:
re.search(r'hello', 'HELLO world', re.IGNORECASE)
```

Same effect, two spellings — like `count=1` vs positional `1` from a few questions back, but here **both** forms are fully supported (no deprecation).

---

### `re.IGNORECASE` / `(?i)` — Case Doesn't Matter

Normally regex is case-sensitive:

```python
re.search(r'hello', 'HELLO world')                    # → None
re.search(r'hello', 'HELLO world', re.IGNORECASE)      # → matches!
```

Remember `[Hh]e` from your Doctor Who example? `(?i)` makes that trick **unnecessary** — instead of manually listing every capitalization, one flag handles it all:

```python
re.findall(r'(?i)\bhe\b', 'He said hi. HE left. he returned.')
# → ['He', 'HE', 'he']       all three, regardless of case
```

---

### `re.MULTILINE` / `(?m)` — `^` and `$` Work Per-Line

This is exactly the **gotcha from your `\A`/`\Z` question**! Normally `^` and `$` only anchor to the whole string's edges:

```python
text = "first line\nsecond line\nthird line"

re.findall(r'^\w+', text)
# → ['first']                  ← only ONE match, string-start only
```

With `re.MULTILINE`, `^` and `$` additionally anchor at **every line break**:

```python
re.findall(r'^\w+', text, re.MULTILINE)
# → ['first', 'second', 'third']   ← now matches at each line's start!
```

This is the flag that **causes** the very difference between `^`/`\A` you asked about earlier — `\A` never changes regardless of this flag, but `^` does.

---

### `re.DOTALL` / `(?s)` — Dot Also Matches Newlines

Recall: `.` matches *any character* — except newline, by default:

```python
text = "line one\nline two"

re.search(r'one.two', text)              # → None — the \n blocks the dot!
re.search(r'one.two', text, re.DOTALL)   # → matches — dot now eats \n too
```

Useful when you want `.*` to span across multiple lines — like grabbing everything between two markers in a multi-line document.

---

### Cheat Sheet

| Flag | Inline | Changes | Without it | With it |
|---|---|---|---|---|
| `re.IGNORECASE` | `(?i)` | letter case | `Hello` ≠ `hello` | `Hello` = `hello` |
| `re.MULTILINE` | `(?m)` | `^` `$` scope | anchor whole string | anchor every line |
| `re.DOTALL` | `(?s)` | `.` scope | `.` skips `\n` | `.` matches `\n` too |

---

### Combining Flags

You can stack multiple flags with `|` (the same bitwise-OR you've seen in other contexts):

```python
re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
```

---

### The One-Sentence Summary

> Flags are **global switches** that relax or extend the default matching rules for the whole pattern — `IGNORECASE` erases case sensitivity, `MULTILINE` makes `^`/`$` fire at every line, and `DOTALL` lets `.` swallow newlines too. Set them either inline as `(?i)` etc., or as an extra argument to `compile`/`search`/`match`. 🎯

In [26]:
text = "line one\nline two"

print(re.search(r'one.two', text))              # → None — the \n blocks the dot!
print(re.search(r'one.two', text, re.DOTALL))   # → matches — dot now eats \n too

None
None


## Let's Debug This Together

You're right — both printed `None`. Let me trace through exactly why, character by character.

---

### The String, Laid Out

```python
text = "line one\nline two"
```

```
l i n e   o n e \n l i n e   t w o
```

Notice: **there's a space between "one" and the newline** — the text is `"...one\nline two"`, not `"...one\ntwo"`.

---

### What the Pattern Actually Asks For

```python
r'one.two'
```

This means: `o`, `n`, `e`, then **exactly ONE character**, then `t`, `w`, `o`. But between "one" and "two" in your string, there isn't **one** character — there are **five**: `\n`, `l`, `i`, `n`, `e`, ` ` (newline + "line" + space)!

```
one \n l i n e   two
    └──────┬──────┘
        5 characters — but the pattern only allows for 1 (the dot)
```

So **even with `DOTALL`**, the dot still only consumes **one** character. `DOTALL` changes *what kind* of character `.` can match (newline included) — it doesn't change *how many*. That's the bug: my earlier example needed `.*` (zero or more), not `.` (exactly one)!

---

### The Fix — Use `.*` (or a bigger string)

**Option A — fix the pattern** (use `.*` to span the gap):

```python
import re

text = "line one\nline two"

print(re.search(r'one.*two', text))              # still None — . stops at \n
print(re.search(r'one.*two', text, re.DOTALL))   # ✓ now matches!
```

```
None
<re.Match object; span=(5, 18), match='one\nline two'>
```

**Now** you can see `DOTALL` earning its keep: `.*` without the flag refuses to cross `\n` at all; with `DOTALL`, `.*` treats `\n` like any other character and spans right across it.

---

### Option B — Simplify My Original Text to Match the Simple Pattern

If we want to keep `.` (single character) as in my original explanation, the string needs exactly **one** character between "one" and "two":

```python
text2 = "one\ntwo"     # exactly one char (\n) between them

print(re.search(r'one.two', text2))              # → None      dot blocks \n
print(re.search(r'one.two', text2, re.DOTALL))   # → matches!  dot allows \n
```

```
None
<re.Match object; span=(0, 7), match='one\ntwo'>
```

Now the demo works exactly as originally intended — apologies for that mismatch, good catch spotting it!

---

### The Underlying Lesson (Worth Keeping)

> `.` = **exactly one** character (newline-excluded, unless `DOTALL`)
> `.*` = **any number** of characters (each individually still subject to the newline rule)
>
> `DOTALL` only redefines what counts as "any character" — it doesn't turn a single-character `.` into a multi-character wildcard. If your goal is *"span across a newline,"* you almost always want `.*` (or `.+`), not a bare `.`. 🎯

Thanks for testing the code rather than taking my example on faith — that's exactly the instinct that's caught real issues throughout this conversation (the `sqrt` bug, the `filter`-returns-list claim, the flowchart arrows). Keep running the code!

When giving multiple flags to the compile function, the flags can be separated with the | sign. For example, re.compile(pattern, re.MULTILINE | re.DOTALL). This is equal to re.compile('(?m)(?s)' + pattern).